In [1]:
print('hi')

hi


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="PdufbKJSUUS5xZED1Vrw")
project = rf.workspace("rahul-kishore-gorai").project("mot-ma3bf-5b96r")
version = project.version(1)
dataset = version.download("yolo26")
                

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 101.1 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0,


Extracting Dataset Version Zip to MOT-1 in yolo26:: 100%|██████████| 18346/18346 [00:04<00:00, 4575.32it/s] 


In [3]:
!ls MOT-1

data.yaml  README.dataset.txt  README.roboflow.txt  train  valid


In [4]:
!cat MOT-1/data.yaml

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 1
names: ['people']

roboflow:
  workspace: rahul-kishore-gorai
  project: mot-ma3bf-5b96r
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/rahul-kishore-gorai/mot-ma3bf-5b96r/dataset/1

In [5]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.1 MB/s eta 0:00:00


In [6]:
import os
import cv2
import random
import torch
import numpy as np
import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
from ultralytics import YOLO
from ultralytics.data.augment import Albumentations
from ultralytics.utils import LOGGER, colorstr

class AddScanLines(ImageOnlyTransform):
    """
    Adds simulated scan lines (LED strips) to an image, mimicking the effect of photographing a screen.
    """
    def __init__(self, line_density=10, line_thickness_range=(1, 5), opacity_range=(0.1, 0.3), always_apply=False, p=1.0):
        super(AddScanLines, self).__init__(p=p)
        self.line_density = line_density
        self.line_thickness_range = line_thickness_range
        self.opacity_range = opacity_range

    def apply(self, image, **params):
        height, width, _ = image.shape
        image_with_lines = image.copy()
        num_lines = random.randint(5, self.line_density)

        for _ in range(num_lines):
            y_start = random.randint(0, height - 1)
            y_end = y_start + random.randint(1, 5)  
            
            line_thickness = random.randint(*self.line_thickness_range)
            opacity = random.uniform(*self.opacity_range)
            line_color = (random.randint(50, 150), random.randint(50, 150), random.randint(50, 150))
            
            overlay = image_with_lines.copy()
            cv2.line(overlay, (0, y_start), (width, y_end), line_color, line_thickness)
            cv2.addWeighted(overlay, opacity, image_with_lines, 1 - opacity, 0, image_with_lines)

        return image_with_lines
    
    def get_transform_init_args_names(self):
        return ("line_density", "line_thickness_range", "opacity_range")
    
class AddMoirePattern(ImageOnlyTransform):
    """
    Adds synthetic moiré patterns to the image by overlaying a rotated and shifted grid pattern.
    """
    def __init__(self, grid_density=20, rotation_range=(-15, 15), opacity_range=(0.05, 0.2), always_apply=False, p=1.0):
        super(AddMoirePattern, self).__init__(p)
        self.grid_density = grid_density
        self.rotation_range = rotation_range
        self.opacity_range = opacity_range

    def apply(self, image, **params):
        image = image.copy() 
        height, width, _ = image.shape
        
        grid = np.zeros((height, width), dtype=np.uint8)
        step_size = max(1, min(width, height) // self.grid_density)
        
        for i in range(0, width, step_size):
            cv2.line(grid, (i, 0), (i, height), 255, 1)
        for i in range(0, height, step_size):
            cv2.line(grid, (0, i), (width, i), 255, 1)
        
        rotation_angle = random.uniform(*self.rotation_range)
        rotation_matrix = cv2.getRotationMatrix2D((width // 2, height // 2), rotation_angle, 1)
        rotated_grid = cv2.warpAffine(grid, rotation_matrix, (width, height))
        
        opacity = random.uniform(*self.opacity_range)
        grid_rgb = cv2.cvtColor(rotated_grid, cv2.COLOR_GRAY2BGR)
        
        image_with_moire = cv2.addWeighted(image, 1 - opacity, grid_rgb, opacity, 0)
        return image_with_moire   
    
    def get_transform_init_args_names(self):
        return ("grid_density", "rotation_range", "opacity_range")


# UPDATED SIGNATURE: Added transforms=None and **kwargs to absorb Ultralytics updates
def custom_albumentations_init(self, p=1.0, transforms=None, **kwargs):
    """
    Initialize the transform object for YOLO bbox formatted params.
    This replaces Ultralytics default Albumentations init.
    """
    self.p = p
    self.transform = None
    self.contains_spatial = True
    prefix = colorstr("albumentations: ")
    try:
        print("Attempting to compose custom Albumentations pipeline...")
        
        T = [
            # Pixel-level transforms 
            A.Blur(blur_limit=3, p=0.01),
            A.ChannelDropout(p=0.3),
            A.ChannelShuffle(p=0.3),
            A.ChromaticAberration(p=0.2),
            A.CLAHE(p=0.1),
            A.ColorJitter(p=0.2),
            A.Defocus(p=0.0001),
            A.Emboss(p=0.01),
            A.FancyPCA(alpha=0.1, p=0.2),
            A.GaussianBlur(blur_limit=(1, 3), p=0.01),
            A.GaussNoise(var_limit=(10, 20), p=0.01),
            A.GlassBlur(sigma=0.6, max_delta=3, iterations=1, p=0.001),
            A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=50, val_shift_limit=10, p=0.2),
            A.ISONoise(p=0.3),
            A.InvertImg(p=0.01),
            A.MedianBlur(blur_limit=3, p=0.01),
            A.MotionBlur(blur_limit=3, p=0.05),
            A.MultiplicativeNoise(p=0.1),
            A.PlanckianJitter(p=0.2),
            A.RandomBrightnessContrast(p=0.1),
            A.RandomFog(p=0.1),
            A.RandomGamma(gamma_limit=(80, 120), p=0.1),
            A.RandomToneCurve(p=0.2),
            A.RingingOvershoot(p=0.005),
            A.RGBShift(r_shift_limit=10, g_shift_limit=10, b_shift_limit=10, p=0.2),
            A.ToSepia(p=0.1),
            A.Sharpen(p=0.1),
            A.Spatter(p=0.005),
            A.Superpixels(p=0.001),
            A.ToGray(p=0.3),
            A.UnsharpMask(p=0.05),
            
            # Spatial transforms
            A.HorizontalFlip(p=0.3),              
            A.VerticalFlip(p=0.3),                    
            A.RandomCrop(width=300, height=300, p=0.1), 
            A.Rotate(limit=90, p=0.1),                
            
            # Custom transforms
            AddScanLines(line_density=105, line_thickness_range=(1, 4), opacity_range=(0.1, 0.3), p=0.1),
            AddMoirePattern(grid_density=30, rotation_range=(-10, 10), opacity_range=(0.05, 0.15), p=0.2),
            
            # Final Resize
            A.Resize(height=640, width=640, p=1.0),
        ]
        
        self.transform = A.Compose(
            T, 
            bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.3)
        )
        
        print("Transformations successfully composed.") 
        LOGGER.info(prefix + ", ".join(f"{x}".replace("always_apply=False, ", "") for x in T if getattr(x, 'p', 1.0)))
        
    except ImportError:  
        print("Albumentations not installed. Skipping")
        pass
    except Exception as e:
        LOGGER.info(f"{prefix}Error: {e}")

# 1. Apply the monkey patch to override Ultralytics Albumentations
Albumentations.__init__ = custom_albumentations_init


if __name__ == '__main__':
    # 2. Load model
    model = YOLO('yolo26s.pt')

    # 3. Train
    results = model.train(
        data='MOT-1/data.yaml', 
        freeze=list(range(12)),
        epochs=100, 
        batch=56, 
        imgsz=640, 
        workers=0,
        device='0,1',
        patience=10
    )

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=56, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=MOT-1/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fracti

In [7]:
!zip -r runs_freeze.zip runs/detect

  adding: runs/detect/ (stored 0%)
  adding: runs/detect/train/ (stored 0%)
  adding: runs/detect/train/results.csv (deflated 67%)
  adding: runs/detect/train/BoxF1_curve.png (deflated 18%)
  adding: runs/detect/train/val_batch0_pred.jpg (deflated 5%)
  adding: runs/detect/train/confusion_matrix_normalized.png (deflated 36%)
  adding: runs/detect/train/BoxP_curve.png (deflated 19%)
  adding: runs/detect/train/val_batch1_labels.jpg (deflated 5%)
  adding: runs/detect/train/BoxR_curve.png (deflated 17%)
  adding: runs/detect/train/val_batch1_pred.jpg (deflated 4%)
  adding: runs/detect/train/val_batch2_labels.jpg (deflated 5%)
  adding: runs/detect/train/train_batch1.jpg (deflated 3%)
  adding: runs/detect/train/train_batch0.jpg (deflated 3%)
  adding: runs/detect/train/args.yaml (deflated 52%)
  adding: runs/detect/train/confusion_matrix.png (deflated 33%)
  adding: runs/detect/train/BoxPR_curve.png (deflated 21%)
  adding: runs/detect/train/val_batch2_pred.jpg (deflated 4%)
  adding: r

In [8]:
import os
import cv2
import random
import torch
import numpy as np
import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
from ultralytics import YOLO
from ultralytics.data.augment import Albumentations
from ultralytics.utils import LOGGER, colorstr

class AddScanLines(ImageOnlyTransform):
    """
    Adds simulated scan lines (LED strips) to an image, mimicking the effect of photographing a screen.
    """
    def __init__(self, line_density=10, line_thickness_range=(1, 5), opacity_range=(0.1, 0.3), always_apply=False, p=1.0):
        super(AddScanLines, self).__init__(p=p)
        self.line_density = line_density
        self.line_thickness_range = line_thickness_range
        self.opacity_range = opacity_range

    def apply(self, image, **params):
        height, width, _ = image.shape
        image_with_lines = image.copy()
        num_lines = random.randint(5, self.line_density)

        for _ in range(num_lines):
            y_start = random.randint(0, height - 1)
            y_end = y_start + random.randint(1, 5)  
            
            line_thickness = random.randint(*self.line_thickness_range)
            opacity = random.uniform(*self.opacity_range)
            line_color = (random.randint(50, 150), random.randint(50, 150), random.randint(50, 150))
            
            overlay = image_with_lines.copy()
            cv2.line(overlay, (0, y_start), (width, y_end), line_color, line_thickness)
            cv2.addWeighted(overlay, opacity, image_with_lines, 1 - opacity, 0, image_with_lines)

        return image_with_lines
    
    def get_transform_init_args_names(self):
        return ("line_density", "line_thickness_range", "opacity_range")
    
class AddMoirePattern(ImageOnlyTransform):
    """
    Adds synthetic moiré patterns to the image by overlaying a rotated and shifted grid pattern.
    """
    def __init__(self, grid_density=20, rotation_range=(-15, 15), opacity_range=(0.05, 0.2), always_apply=False, p=1.0):
        super(AddMoirePattern, self).__init__(p)
        self.grid_density = grid_density
        self.rotation_range = rotation_range
        self.opacity_range = opacity_range

    def apply(self, image, **params):
        image = image.copy() 
        height, width, _ = image.shape
        
        grid = np.zeros((height, width), dtype=np.uint8)
        step_size = max(1, min(width, height) // self.grid_density)
        
        for i in range(0, width, step_size):
            cv2.line(grid, (i, 0), (i, height), 255, 1)
        for i in range(0, height, step_size):
            cv2.line(grid, (0, i), (width, i), 255, 1)
        
        rotation_angle = random.uniform(*self.rotation_range)
        rotation_matrix = cv2.getRotationMatrix2D((width // 2, height // 2), rotation_angle, 1)
        rotated_grid = cv2.warpAffine(grid, rotation_matrix, (width, height))
        
        opacity = random.uniform(*self.opacity_range)
        grid_rgb = cv2.cvtColor(rotated_grid, cv2.COLOR_GRAY2BGR)
        
        image_with_moire = cv2.addWeighted(image, 1 - opacity, grid_rgb, opacity, 0)
        return image_with_moire   
    
    def get_transform_init_args_names(self):
        return ("grid_density", "rotation_range", "opacity_range")


# UPDATED SIGNATURE: Added transforms=None and **kwargs to absorb Ultralytics updates
def custom_albumentations_init(self, p=1.0, transforms=None, **kwargs):
    """
    Initialize the transform object for YOLO bbox formatted params.
    This replaces Ultralytics default Albumentations init.
    """
    self.p = p
    self.transform = None
    self.contains_spatial = True
    prefix = colorstr("albumentations: ")
    try:
        print("Attempting to compose custom Albumentations pipeline...")
        
        T = [
            # Pixel-level transforms 
            A.Blur(blur_limit=3, p=0.01),
            A.ChannelDropout(p=0.3),
            A.ChannelShuffle(p=0.3),
            A.ChromaticAberration(p=0.2),
            A.CLAHE(p=0.1),
            A.ColorJitter(p=0.2),
            A.Defocus(p=0.0001),
            A.Emboss(p=0.01),
            A.FancyPCA(alpha=0.1, p=0.2),
            A.GaussianBlur(blur_limit=(1, 3), p=0.01),
            A.GaussNoise(var_limit=(10, 20), p=0.01),
            A.GlassBlur(sigma=0.6, max_delta=3, iterations=1, p=0.001),
            A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=50, val_shift_limit=10, p=0.2),
            A.ISONoise(p=0.3),
            A.InvertImg(p=0.01),
            A.MedianBlur(blur_limit=3, p=0.01),
            A.MotionBlur(blur_limit=3, p=0.05),
            A.MultiplicativeNoise(p=0.1),
            A.PlanckianJitter(p=0.2),
            A.RandomBrightnessContrast(p=0.1),
            A.RandomFog(p=0.1),
            A.RandomGamma(gamma_limit=(80, 120), p=0.1),
            A.RandomToneCurve(p=0.2),
            A.RingingOvershoot(p=0.005),
            A.RGBShift(r_shift_limit=10, g_shift_limit=10, b_shift_limit=10, p=0.2),
            A.ToSepia(p=0.1),
            A.Sharpen(p=0.1),
            A.Spatter(p=0.005),
            A.Superpixels(p=0.001),
            A.ToGray(p=0.3),
            A.UnsharpMask(p=0.05),
            
            # Spatial transforms
            A.HorizontalFlip(p=0.3),              
            A.VerticalFlip(p=0.3),                    
            A.RandomCrop(width=300, height=300, p=0.1), 
            A.Rotate(limit=90, p=0.1),                
            
            # Custom transforms
            AddScanLines(line_density=105, line_thickness_range=(1, 4), opacity_range=(0.1, 0.3), p=0.1),
            AddMoirePattern(grid_density=30, rotation_range=(-10, 10), opacity_range=(0.05, 0.15), p=0.2),
            
            # Final Resize
            A.Resize(height=640, width=640, p=1.0),
        ]
        
        self.transform = A.Compose(
            T, 
            bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.3)
        )
        
        print("Transformations successfully composed.") 
        LOGGER.info(prefix + ", ".join(f"{x}".replace("always_apply=False, ", "") for x in T if getattr(x, 'p', 1.0)))
        
    except ImportError:  
        print("Albumentations not installed. Skipping")
        pass
    except Exception as e:
        LOGGER.info(f"{prefix}Error: {e}")

# 1. Apply the monkey patch to override Ultralytics Albumentations
Albumentations.__init__ = custom_albumentations_init


if __name__ == '__main__':
    # 2. Load model
    model = YOLO('runs/detect/train/weights/best.pt')

    # 3. Train
    results = model.train(
        data='MOT-1/data.yaml',
        epochs=50, 
        batch=32, 
        imgsz=640, 
        workers=0,
        device='0,1',
        patience=10
    )

Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=MOT-1/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/train/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nb

In [9]:
!zip -r runs.zip runs/detect

  adding: runs/detect/ (stored 0%)
  adding: runs/detect/train/ (stored 0%)
  adding: runs/detect/train/results.csv (deflated 67%)
  adding: runs/detect/train/BoxF1_curve.png (deflated 18%)
  adding: runs/detect/train/val_batch0_pred.jpg (deflated 5%)
  adding: runs/detect/train/confusion_matrix_normalized.png (deflated 36%)
  adding: runs/detect/train/BoxP_curve.png (deflated 19%)
  adding: runs/detect/train/val_batch1_labels.jpg (deflated 5%)
  adding: runs/detect/train/BoxR_curve.png (deflated 17%)
  adding: runs/detect/train/val_batch1_pred.jpg (deflated 4%)
  adding: runs/detect/train/val_batch2_labels.jpg (deflated 5%)
  adding: runs/detect/train/train_batch1.jpg (deflated 3%)
  adding: runs/detect/train/train_batch0.jpg (deflated 3%)
  adding: runs/detect/train/args.yaml (deflated 52%)
  adding: runs/detect/train/confusion_matrix.png (deflated 33%)
  adding: runs/detect/train/BoxPR_curve.png (deflated 21%)
  adding: runs/detect/train/val_batch2_pred.jpg (deflated 4%)
  adding: r